# 去噪扩散概率模型 Denoising Diffusion Probabilistic Models

```bibtex
@article{ho2020denoising,
  title={Denoising Diffusion Probabilistic Models},
  author={Jonathan Ho and Ajay Jain and Pieter Abbeel},
  year={2020},
  journal={arXiv preprint arxiv:2006.11239}
}
```

笔记内容参考博客:
- [狗都能看懂的DDPM论文详解 CSDN原文链接](https://blog.csdn.net/weixin_42392454/article/details/137458318)
- [扩散模型之DDPM 知乎内容](https://zhuanlan.zhihu.com/p/563661713)


# 论文解读 Denoising Diffusion Probabilistic Models

## 0 摘要 Abstract

该研究提出利用扩散概率模型（一类受非平衡热力学启发的潜变量模型）实现高质量图像合成：
1. 核心方法：通过一种加权变分边界训练模型，该边界源于扩散概率模型与基于朗之万动力学的去噪分数匹配之间的新关联；
2. 模型特性：天然支持渐进式有损解压方案，可视为自回归解码的泛化形式；
3. 实验成果：在无条件 CIFAR10 数据集上，取得 9.46 的 Inception 分数和 3.17 的最先进 FID 分数；在 256x256 分辨率的 LSUN 数据集上，样本质量与 ProgressiveGAN 相当；
4. 资源获取：相关实现已开源，地址为: [diffusion](https://github.com/hojonathanho/diffusion)

## 2 背景 Background

1. **模型定义**：扩散模型是一类潜变量模型，形式为 $p_\theta(\mathbf{x}_0) \coloneqq \int p_\theta(\mathbf{x}_{0:T}) d\mathbf{x}_{1:T}$，其中 $\mathbf{x}_1,\dots,\mathbf{x}_T$ 是与数据 $\mathbf{x}_0$ 维度相同的潜变量，核心包含两个 Markov 链过程：
   - **反向过程（生成过程）**：联合概率 $p_\theta(\mathbf{x}_{0:T})$，被定义为一个以 $p(\mathbf{x}_T)=\mathcal{N}(\mathbf{x}_T;\mathbf{0},\mathbf{I})$ 为起点的具有学习参数的高斯转换马尔可夫链。
    $$\begin{aligned}
      p_\theta(\mathbf{x}_{0:T}) &\coloneqq p(\mathbf{x}_T)\prod_{t=1}^T p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t), \qquad 
      p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t) \coloneqq \mathcal{N}(\mathbf{x}_{t-1}; {\boldsymbol{\mu}}_\theta(\mathbf{x}_t, t), {\boldsymbol{\Sigma}}_\theta(\mathbf{x}_t, t)) \qquad (1)
    \end{aligned}$$
    <a id="equation_1"> </a>
   - **正向过程（扩散过程）**：近似后验分布 $q(\mathbf{x}_{1:T}|\mathbf{x}_0)$，按固定方差调度 $\beta_1,\dots,\beta_T$ 向数据逐步添加高斯噪声，且任意时刻 $t$ 的边际分布 $q(\mathbf{x}_t|\mathbf{x}_0)$ 可通过 $\bar\alpha_t = \prod_{s=1}^t (1-\beta_s)$ 表示为闭形式正态分布。
    $$\begin{aligned}
    q(\mathbf{x}_{1:T} | \mathbf{x}_0) &\coloneqq \prod_{t=1}^T q(\mathbf{x}_t | \mathbf{x}_{t-1} ), \qquad 
    q(\mathbf{x}_t|\mathbf{x}_{t-1}) \coloneqq \mathcal{N}(\mathbf{x}_t;\sqrt{1-\beta_t}\mathbf{x}_{t-1},\beta_t \mathbf{I}) \qquad (2)
    \end{aligned}$$
    <a id="equation_2"> </a>

2. **训练优化**：

训练通过优化负对数似然的常规变分界限进行：

$$\begin{aligned}
\mathbb{E}\left[-\log p_{\boldsymbol{\theta}}(\mathbf{x}_0)\right] \leq \mathbb{E}_{q}\!\left[ - \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T} | \mathbf{x}_0)} \right]
  = \mathbb{E}_q\bigg[ -\log p(\mathbf{x}_T) - \sum_{t \geq 1} \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1} | \mathbf{x}_t)}{q(\mathbf{x}_t|\mathbf{x}_{t-1})} \bigg] \coloneqq L \qquad (3)
\end{aligned}$$
<a id="equation_3"> </a>

正向过程方差$\beta_t$可以通过重新参数化来学习，或作为超参数保持不变，而反向过程的表达能力部分取决于$p_\theta(x_{t-1}|x_t)$中高斯条件的选择，因为当$\beta_t$较小时，两个过程具有相同的功能形式。正向过程的一个显著特性是它允许在任意时间步长$t$以封闭形式进行采样$x_t$: 使用符号$a_t := 1 - \beta_t$和 $\bar{\alpha}_t \coloneqq \prod_{s=1}^t \alpha_s$，我们有
$$\begin{aligned}
  q(\mathbf{x}_t|\mathbf{x}_0) = \mathcal{N}(\mathbf{x}_t; \sqrt{\bar\alpha_t}\mathbf{x}_0, (1-\bar\alpha_t)\mathbf{I}) \qquad (4)
\end{aligned}$$
<a id="equation_4"> </a>

因此，可以通过随机梯度下降优化$L$中的随机项来实现高效的训练。进一步的改进来自于通过将$L$[公式3](#equation_3)重写为[公式5](#equation_5)：
$$\begin{aligned}
\mathbb{E}_q \bigg[ \underbrace{D_{\mathrm{KL}}\!\left(q(\mathbf{x}_T|\mathbf{x}_0) ~ \| ~ p(\mathbf{x}_T)\right)}_{L_T} + \sum_{t > 1} \underbrace{D_{\mathrm{KL}}\!\left(q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0) ~ \| ~ p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)\right)}_{L_{t-1}} + \underbrace{-\log p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)}_{L_0} \bigg] \qquad (5)
\end{aligned}$$
<a id="equation_5"> </a>

（详见附录 A。这些术语上的标签在第三节中使用。）[公式5](#equation_5) 使用 KL 散度来直接比较$p_\theta(x_{t-1}|x_t)$与正向过程后验，当以$x_0$为条件时，这是可行的：

$$\begin{aligned}
q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0) &=  \mathcal{N}(\mathbf{x}_{t-1}; \tilde{\boldsymbol{\mu}}_t(\mathbf{x}_t, \mathbf{x}_0), \tilde{\beta}_t \mathbf{I}),  \qquad (6) \\
\text{where}\quad \tilde{\boldsymbol{\mu}}_t(\mathbf{x}_t, \mathbf{x}_0) &\coloneqq \frac{\sqrt{\bar\alpha_{t-1}}\beta_t }{1-\bar\alpha_t}\mathbf{x}_0 + \frac{\sqrt{\alpha_t}(1- \bar\alpha_{t-1})}{1-\bar\alpha_t} \mathbf{x}_t \quad \text{and} \quad
\tilde{\beta}_t \coloneqq \frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\beta_t  \qquad(7)
\end{aligned}$$
<a id="equation_7"> </a>


因此，[公式5](#equation_5)中的所有 KL 散度都是高斯分布之间的比较，所以它们可以用闭式表达式以 Rao-Blackwell 化的方式计算，而不是用高方差蒙特卡洛估计。

- 目标是最小化负对数似然的变分上界 $L$，原始形式可拆解为 KL 散度项与对数概率项的组合；
- 关键改进：通过重写变分上界，将各项转化为高斯分布间的 KL 散度（含 $L_T$、$L_{t-1}$、$L_0$ 三项），其中 $q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0)$ 具有可解析的均值 $\tilde{\boldsymbol{\mu}}_t$ 和方差 $\tilde\beta_t$，因此所有 KL 散度可通过闭形式计算，无需高方差蒙特卡洛估计，支持高效随机梯度下降训练；
- 正向过程方差 $\beta_t$ 可作为超参数固定或通过重参数化学习，反向过程采用高斯条件分布以保证与正向过程（$\beta_t$ 较小时）的函数形式一致性，确保模型表达能力。


3. **核心特性**：正向过程可在任意时刻 $t$ 直接采样 $\mathbf{x}_t$，反向过程通过学习从噪声逐步恢复数据，二者结合构成扩散模型的核心框架，其关系可通过有向图模型（如[图2](#fig_2)）直观表示。

<a id="fig_2" />

<div style="background-color:white; padding:10px; border-radius:5px; width: 80%; margin:auto; text-align: center;">
    <image src="./assets/pgm_diagram_xarrow_small.png"/>
    <p style="font-size:12px; color:gray;">图 2. 论文中考虑的有向图模型</p>
</div>

__公式推理__

<!-- - [公式3](#equation_3) 中的不等式来源于 KL 散度的非负性，等式则源于正向和反向过程的定义。具体推导如下： -->
> ### 公式3 中不等式和等式的推导步骤
> #### 第一个小于号根据KL散度的非负性成立:
> $$\begin{aligned}
>  D_{\text{KL}}\big(q\parallel p_\theta\big) &\ge 0 \\
> \Rightarrow D_{\text{KL}}\big(q(\mathbf{x}_{1:T}|\mathbf{x}_0)\parallel p_\theta(\mathbf{x}_{1:T}|\mathbf{x}_0)\big)  &\ge 0 \\ 
> \Rightarrow \mathbb{E}_q\left[\log\frac{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}{p_\theta(\mathbf{x}_{1:T}|\mathbf{x}_0)}\right] &\ge 0 \\
> \Rightarrow \mathbb{E}_{q}\big[\log q(\mathbf{x}_{1:T}|\mathbf{x}_0) - \log p_\theta(\mathbf{x}_{1:T}|\mathbf{x}_0)\big] &\ge 0 \\
> \Rightarrow \mathbb{E}_{q}\big[\log q(\mathbf{x}_{1:T}|\mathbf{x}_0) - \log p_\theta(\mathbf{x}_{1:T}|\mathbf{x}_0) - \log p_\theta(\mathbf{x}_0)\big] &\ge \mathbb{E} \big[ - \log p_\theta(\mathbf{x}_0) \big] \\
> \Rightarrow \mathbb{E}_{q}\big[\log q(\mathbf{x}_{1:T}|\mathbf{x}_0) - \log p_\theta(\mathbf{x}_{0:T})\big] &\ge \mathbb{E} \big[ - \log p_\theta(\mathbf{x}_0) \big] \\
> \Rightarrow \mathbb{E}_{q}\!\left[ - \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T} | \mathbf{x}_0)} \right] &\ge \mathbb{E} \big[ - \log p_\theta(\mathbf{x}_0) \big]
> \end{aligned}$$
> #### 第二个等于号根据上述两个过程的定义成立:
> $$\begin{aligned}
> \mathbb{E}_{q}\!\left[ - \log \frac{p_{\boldsymbol{\theta}}(\mathbf{x}_{0:T})}{q(\mathbf{x}_{1:T} | \mathbf{x}_0)} \right] &= \mathbb{E}_{q}\!\left[ - \log \frac{p(\mathbf{x}_T)\prod_{t=1}^T p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t)}{\prod_{t=1}^T q(\mathbf{x}_t | \mathbf{x}_{t-1} )} \right] \\
> &= \mathbb{E}_{q}\!\left[ - \log p(\mathbf{x}_T) - \sum_{t=1}^T \log \frac{p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_t | \mathbf{x}_{t-1} )} \right] \\
> \end{aligned}$$

> ### 公式3 到 公式5 的推导步骤
> 1. **扩散过程**：$q(\mathbf{x}_{1:T}|\mathbf{x}_0) = \prod_{t=1}^T q(\mathbf{x}_t|\mathbf{x}_{t-1})$，其中 $q(\mathbf{x}_t|\mathbf{x}_{t-1})$ 是前向扩散的高斯转移核。
> 2. **KL散度定义**：$D_{\text{KL}}(q\|p) = \mathbb{E}_q\left[\log\frac{q}{p}\right]$，因此 $\mathbb{E}_q\left[-\log\frac{p}{q}\right] = D_{\text{KL}}(q\|p)$。
> 3. **贝叶斯定理**：$q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0) = \frac{q(\mathbf{x}_t|\mathbf{x}_{t-1}) q(\mathbf{x}_{t-1}|\mathbf{x}_0)}{q(\mathbf{x}_t|\mathbf{x}_0)}$。
> 4. **马尔可夫性**：反向模型 $p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t)$ 仅依赖 $\mathbf{x}_t$，与 $\mathbf{x}_0$ 无关。
> #### 步骤1：拆分式3的求和项
> 式3的ELBO为
> $$L = \mathbb{E}_q\left[ -\log p(\mathbf{x}_T) - \sum_{t=1}^T \log\frac{p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_t|\mathbf{x}_{t-1})} \right]$$
> 将求和项拆分为 $t=T$ 项和 $t=1$ 到 $T-1$ 的项：
> $$ L = \mathbb{E}_q\left[ -\log p(\mathbf{x}_T) - \log\frac{p_\theta(\mathbf{x}_{T-1}|\mathbf{x}_T)}{q(\mathbf{x}_T|\mathbf{x}_{T-1})} - \sum_{t=1}^{T-1} \log\frac{p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_t|\mathbf{x}_{t-1})} \right]$$
> 其中$x_0$对应$t=0$，$x_T$是最终噪声。
> 
> #### 步骤2：处理 $t=T$ 项（对应 $L_T$）
> 对 $t=T$ 项，结合**联合分布拆分** $q(\mathbf{x}_{1:T}|\mathbf{x}_0) = q(\mathbf{x}_T|\mathbf{x}_0) \cdot q(\mathbf{x}_{1:T-1}|\mathbf{x}_T,\mathbf{x}_0)$，对期望做**变量替换**：
> 将期望 $\mathbb{E}_{q(\mathbf{x}_{1:T}|\mathbf{x}_0)}$ 拆分为 $\mathbb{E}_{q(\mathbf{x}_T|\mathbf{x}_0)} \cdot \mathbb{E}_{q(\mathbf{x}_{1:T-1}|\mathbf{x}_T,\mathbf{x}_0)}$。
> 观察第一项 $-\log p(\mathbf{x}_T) - \log\frac{p_\theta(\mathbf{x}_{T-1}|\mathbf{x}_T)}{q(\mathbf{x}_T|\mathbf{x}_{T-1})}$，利用**贝叶斯定理**替换 $q(\mathbf{x}_T|\mathbf{x}_{T-1})$：
> $$
> q(\mathbf{x}_T|\mathbf{x}_{T-1}) = \frac{q(\mathbf{x}_{T-1}|\mathbf{x}_T,\mathbf{x}_0) q(\mathbf{x}_T|\mathbf{x}_0)}{q(\mathbf{x}_{T-1}|\mathbf{x}_0)}
> $$
> 代入后，对 $\mathbf{x}_{1:T-1}$ 求期望时，$\log p_\theta(\mathbf{x}_{T-1}|\mathbf{x}_T)$ 和 $q(\mathbf{x}_{T-1}|\mathbf{x}_T,\mathbf{x}_0)$ 的积分/求和会抵消，最终剩余：
> $$
> \mathbb{E}_{q(\mathbf{x}_T|\mathbf{x}_0)}\left[ -\log\frac{p(\mathbf{x}_T)}{q(\mathbf{x}_T|\mathbf{x}_0)} \right] = D_{\text{KL}}\left(q(\mathbf{x}_T|\mathbf{x}_0) \| p(\mathbf{x}_T)\right) = L_T
> $$
> #### 步骤3：处理 $1 \leq t \leq T-1$ 项（对应 $\sum_{t>1} L_{t-1}$）
> 对 $t \in [1, T-1]$，将求和式改写为 $t'=t-1 \in [0, T-2]$，则 $\sum_{t=1}^{T-1} \cdot = \sum_{t'>0}^{T-2} \cdot$（即 $t>1$ 等价于 $t' \geq 1$）。
> 利用**KL散度定义**和**贝叶斯定理**，对每一项：
> $$ -\log\frac{p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_t|\mathbf{x}_{t-1})} = -\log p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t) + \log q(\mathbf{x}_t|\mathbf{x}_{t-1}) $$
> 结合 $q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0) = \frac{q(\mathbf{x}_t|\mathbf{x}_{t-1}) q(\mathbf{x}_{t-1}|\mathbf{x}_0)}{q(\mathbf{x}_t|\mathbf{x}_0)}$，变形得：
> $$ \log q(\mathbf{x}_t|\mathbf{x}_{t-1}) = \log q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0) + \log q(\mathbf{x}_t|\mathbf{x}_0) - \log q(\mathbf{x}_{t-1}|\mathbf{x}_0) $$
> 代入期望后，相邻项的 $\log q(\mathbf{x}_t|\mathbf{x}_0)$ 和 $\log q(\mathbf{x}_{t-1}|\mathbf{x}_0)$ 会** telescoping 抵消**，最终剩余：
> $$ \sum_{t>1} \mathbb{E}_q\left[ -\log\frac{p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t)}{q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0)} \right] = \sum_{t>1} D_{\text{KL}}\left(q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0) \| p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t)\right) = \sum_{t>1} L_{t-1} $$
> #### 步骤4：处理 $t=1$ 项（对应 $L_0$）
> 当 $t=1$ 时，求和项的剩余部分为：
> $$ -\log\frac{p_\theta(\mathbf{x}_0|\mathbf{x}_1)}{q(\mathbf{x}_1|\mathbf{x}_0)} $$
> 结合 telescoping 抵消后的余项，最终只剩下：
> $$ \mathbb{E}_q\left[ -\log p_\theta(\mathbf{x}_0|\mathbf{x}_1) \right] = L_0 $$
> #### 步骤5：合并所有项
> 将 $L_T$、$\sum_{t>1} L_{t-1}$ 和 $L_0$ 合并，即可得到式 (5)：
> $$ \boxed{L = \mathbb{E}_q \bigg[ \underbrace{D_{\mathrm{KL}}\!\left(q(\mathbf{x}_T|\mathbf{x}_0) ~ \| ~ p(\mathbf{x}_T)\right)}_{L_T} + \sum_{t > 1} \underbrace{D_{\mathrm{KL}}\!\left(q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0) ~ \| ~ p_{\boldsymbol{\theta}}(\mathbf{x}_{t-1}|\mathbf{x}_t)\right)}_{L_{t-1}} + \underbrace{-\log p_{\boldsymbol{\theta}}(\mathbf{x}_0|\mathbf{x}_1)}_{L_0} \bigg]} $$
> #### 关键推导要点
> 1. **Telescoping 和抵消**：是扩散模型ELBO推导的核心技巧，相邻项的分布对数项相互抵消，只保留首尾项。
> 2. **KL散度的变形**：将期望内的对数比直接转化为KL散度，这是变分推断的标准操作。
> 3. **马尔可夫性简化**：反向模型 $p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t)$ 与 $\mathbf{x}_0$ 无关，因此KL散度的后验分布可以简化为 $q(\mathbf{x}_{t-1}|\mathbf{x}_t,\mathbf{x}_0)$。


## 3 扩散模型和去噪自编码器 Diffusion models and denoising autoencoders

扩散模型虽属于一类受限的潜变量模型，但在实现上具备较高自由度，其设计需确定前向过程的方差$\beta_t$、模型架构以及反向过程的高斯分布参数化方式。为指导这些设计选择，研究建立了扩散模型与去噪分数匹配之间的新显式关联，进而推导出扩散模型的简化加权变分下界目标函数。最终，模型的设计合理性通过简洁性与实证结果得以验证，且相关讨论围绕变分下界公式展开。

### 3.1 正向过程和$L_T$ Forward process and $L_T$

实现中固定前向过程方差 $\beta_t$ 为常数而非设为可学习参数，使得近似后验分布 $q$ 无学习参数，训练时 $L_T$ 为常量，可直接忽略。

### 3.2 反向过程和$L_{1: T-1}$ Reverse process and $L_{1:T-1}$

反向过程分布 $p_\theta(\mathbf{x}_{t-1}|\mathbf{x}_t)$ 被设定为高斯分布 $\mathcal{N}(\mathbf{x}_{t-1}; {\boldsymbol{\mu}}_\theta(\mathbf{x}_t, t), {\boldsymbol{\Sigma}}_\theta(\mathbf{x}_t, t))$，具体设计包含两点：
- **协方差参数化**：将协方差矩阵 ${\boldsymbol{\Sigma}}_\theta$ 设为与时间相关的常量 $\sigma_t^2 \mathbf{I}$，实验验证 $\sigma_t^2 = \beta_t$ 和 $\sigma_t^2=\tilde\beta_t=\frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}$ 两种取值效果相近，二者分别对应数据分布的两种极端情况。
- **均值参数化**：对变分下界项 $L_{t-1}$ 推导后发现，最优的均值 ${\boldsymbol{\mu}}_\theta$ 可等价转换为预测噪声 ${\boldsymbol{\epsilon}}$ 的形式，即 ${\boldsymbol{\mu}}_\theta(\mathbf{x}_t, t)=\frac{1}{\sqrt{\alpha_t}}\left( \mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}} {\boldsymbol{\epsilon}}_\theta(\mathbf{x}_t, t) \right)$。该参数化方式将反向过程转化为类朗之万动力学，同时把变分下界目标简化为**类去噪分数匹配目标**。

$$\begin{aligned}
  L_{t-1}
   = \mathbb{E}_{q}\!\left[{ \frac{1}{2\sigma_t^2}\|\tilde{\boldsymbol{\mu}}_t(\mathbf{x}_t,\mathbf{x}_0) - {\boldsymbol{\mu}}_\theta(\mathbf{x}_t, t)\|^2 }\right] + C  \qquad (8)
\end{aligned}$$
<a id="equation_8"> </a>

- 其中$C$是一个不依赖于$\theta$的常数。因此，我们看到$\mu_\theta$的最直接参数化是一个预测$\tilde\mu_t$（正向过程后验均值）的模型。然而，我们可以通过将[公式4](#equation_4)重新参数化为$\mathbf{x}_t(\mathbf{x}_0, {\boldsymbol{\epsilon}}) = \sqrt{\bar\alpha_t}\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}{\boldsymbol{\epsilon}}$(${\boldsymbol{\epsilon}} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$)来进一步扩展[公式8](#equation_8)，并应用正向过程后验[公式7](#equation_7)：

$$\begin{aligned}
L_{t-1} - C &= \mathbb{E}_{x_0, {\boldsymbol{\epsilon}}}\!\left[ \frac{1}{2\sigma_t^2} \left\| \tilde{\boldsymbol{\mu}}_t \left(\mathbf{x}_t(\mathbf{x}_0, {\boldsymbol{\epsilon}}), \frac{1}{\sqrt{\bar\alpha_t}} (\mathbf{x}_t(\mathbf{x}_0, {\boldsymbol{\epsilon}}) - \sqrt{1-\bar\alpha_t}{\boldsymbol{\epsilon}} ) \right) - {\boldsymbol{\mu}}_\theta(\mathbf{x}_t(\mathbf{x}_0, {\boldsymbol{\epsilon}}), t) \right\|^2 \right] \qquad (9) \\
&= \mathbb{E}_{x_0, {\boldsymbol{\epsilon}}}\!\left[ \frac{1}{2\sigma_t^2} \left\| \frac{1}{\sqrt{\alpha_t}} \left(\mathbf{x}_t(\mathbf{x}_0, {\boldsymbol{\epsilon}}) - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \boldsymbol{\epsilon} \right) - {\boldsymbol{\mu}}_\theta(\mathbf{x}_t(\mathbf{x}_0, {\boldsymbol{\epsilon}}), t) \right\|^2 \right] \qquad (10) \\
\end{aligned}$$
<a id="equation_9"> </a>
<a id="equation_10"> </a>

[公式10](#equation_10)表明,$\mu_\theta$必须根据$x_t$预测$\frac{1}{\sqrt{\alpha_t}}\left( \mathbf{x}_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \boldsymbol{\epsilon} \right)$。由于$x_t$作为模型的输入可用，我们可以选择参数化

$$\begin{aligned}
{\boldsymbol{\mu}}_\theta(\mathbf{x}_t, t) = \tilde{\boldsymbol{\mu}}_t\!\left(\mathbf{x}_t,  \frac{1}{\sqrt{\bar\alpha_t}} (\mathbf{x}_t - \sqrt{1-\bar\alpha_t}{\boldsymbol{\epsilon}}_\theta(\mathbf{x}_t)) \right) = \frac{1}{\sqrt{\alpha_t}}\left( \mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}} {\boldsymbol{\epsilon}}_\theta(\mathbf{x}_t, t) \right) \qquad (11)
\end{aligned}$$
<a id="equation_11"> </a>


其中$\epsilon_\theta$是一个函数逼近器，用于从$x_t$预测噪声${\boldsymbol{\epsilon}}$。采样$x_{t-1} \sim p_\theta(x_{t-1}|x_t)$是计算$x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}} {\boldsymbol{\epsilon}}_\theta(\mathbf{x}_t, t) \right) + \sigma_t z$，其中$z \sim \mathcal{N}(0, I)$。完整的采样过程[算法2]()，类似于Langevin动力学，以${\boldsymbol{\epsilon}}_\theta$作为数据密度的学习梯度。此外，在参数化[公式11](#equation_11)下，[公式10](#equation_10)简化为：

$$
\mathbb{E}_{\mathbf{x}_0, \boldsymbol{\epsilon}}\left[ \frac{\beta_t^2}{2\sigma_t^2 \alpha_t (1-\bar\alpha_t)}  \left\| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta\left(\sqrt{\bar\alpha_t} \mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\boldsymbol{\epsilon}, t\right) \right\|^2\right] \qquad (12)
$$
<a id="equation_12"> </a>


这类似于在多个由$t$索引的噪声尺度上进行去噪分数匹配。由于[公式12](#equation_12)等于类似朗之万逆过程的变分界（[公式11](#equation_11) 中的一项），我们看到优化一个类似于去噪分数匹配的目标，相当于使用变分推理来拟合一个类似于朗之万动力学的采样链的有限时间边缘分布。

总而言之，我们可以训练反向过程均值函数逼近器$\mu_\theta$来预测$\tilde{\mu}_t$ ，或者通过修改其参数化方式，我们可以训练它来预测$\boldsymbol{\epsilon}$ 。（也有预测$x_0$的可能性，但我们在实验早期发现这会导致样本质量更差。）我们已经证明， $\boldsymbol{\epsilon}$-预测参数化方式既类似于朗之万动力学，又简化了扩散模型的变分界为类似于去噪分数匹配的目标。尽管它只是$p_\theta(x_{t-1}|x_t)$的另一种参数化方式，我们在第 4 节通过一个消融实验验证了其有效性，在该实验中我们比较了预测$\boldsymbol{\epsilon}$与预测$\tilde{\mu}_t$的效果。

<div style="width: 80%; margin:auto;">
<table style="border: none; ">
<tr>
<td style="border: none">

> #### Algorithm 1 Training
> - **repeat**
>   - $x_0 \sim q(x_0)$
>   - $t \sim \text{Uniform({1, \ldots, T})}$
>   - $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$
>   - Take gradient descent step on
>   - $\nabla_\theta \left\| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta\left(\sqrt{\bar\alpha_t} \mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\boldsymbol{\epsilon}, t\right) \right\|^2$
> - **until** converged

</td>
<td style="border: none">

> #### Algorithm 2 Sampling
> - $x_T \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$
> - **for** $t = T, \ldots, 1$ **do**
>   - $z \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$ if $t > 1$, else $z = \mathbf{0}$
>   - $x_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{1 - \alpha_t}{\sqrt{1-\bar{\alpha}_t}} \boldsymbol{\epsilon}_\theta(x_t, t) \right) + \sigma_t z$
> - **end for**
> - **return** $x_0$

</td>
</tr>
</table>
</div>

**训练与采样算法**
- **训练**：随机采样数据 $\mathbf{x}_0$、时间步 $t$ 和噪声 ${\boldsymbol{\epsilon}}$，通过梯度下降最小化预测噪声 ${\boldsymbol{\epsilon}}_\theta$ 与真实噪声 ${\boldsymbol{\epsilon}}$ 的均方误差。
- **采样**：从标准高斯分布初始化 $\mathbf{x}_T$，逆时间步迭代计算 $\mathbf{x}_{t-1}$，最终得到生成样本 $\mathbf{x}_0$。

**结论**
    噪声预测的参数化方式在理论上与去噪分数匹配等价，且模型效果会通过对比实验（噪声预测 vs 直接预测前向后验均值 $\tilde{\boldsymbol{\mu}}_t$）验证。

### 3.3 数据缩放、逆向过程解码器以及$L_0$ Data scaling, reverse process decoder, and $L_0$

我们假设图像数据由$\{0, 1, \ldots, 255\}$线性缩放到$[-1, 1]$的整数组成。这确保了神经网络反向过程从标准正态先验$p(x_T)$开始，始终在一致缩放的输入上运行。为了获得离散对数似然，我们将反向过程的最后一项设置为从高斯$\mathcal{N}(\mathbf{x}_0; {\boldsymbol{\mu}}_\theta(\mathbf{x}_1, 1), \sigma_1^2 \mathbf{I} )$派生出的独立离散解码器: 

$$\begin{aligned}
\begin{split}
p_\theta(\mathbf{x}_0 | \mathbf{x}_1) &= \prod_{i=1}^D \int_{\delta_{-}(x_0^i)}^{\delta_{+}(x_0^i)} \mathcal{N}(x; \mu_\theta^i(\mathbf{x}_1, 1), \sigma_1^2) \, dx  \\
\delta_{+}(x) &= \begin{cases}
\infty & \text{if}\ x=1 \\
x+\frac{1}{255} & \text{if}\ x < 1
\end{cases}
\qquad \delta_{-}(x) = \begin{cases}
-\infty & \text{if}\ x=-1 \\
x-\frac{1}{255} & \text{if}\ x > -1
\end{cases}
\end{split} \qquad (13)
\end{aligned}$$
<a id="equation_13"> </a>



其中$D$表示数据维度，$i$上标表示提取一个坐标。（可以很直接地集成更强大的解码器，如条件自回归模型，但我们将其留待未来工作。）类似于VAE解码器和自回归模型$[34, 52]$中使用的离散连续分布，我们在此的选择确保变分界是离散数据的无损码长，无需向数据中添加噪声或将缩放操作的雅可比行列式纳入对数似然中。在采样结束时，我们无噪声地显示$\boldsymbol{\mu}_\theta(\mathbf{x}_1, 1)$。

### 3.4 简化的训练目标 Simplified training objective

通过上述定义的逆向过程和解码器，有[公式12](#equation_12)和[公式13](#equation_13)推导出的变分界显然关于$\theta$可微，并准备好用于训练。然而，我们发现对以下变分界变体进行采样能提升采样质量（且实现更简单）:
$$\begin{aligned}
L_\text{simple}(\theta) := \mathbb{E}_{t, \mathbf{x}_0, \boldsymbol{\epsilon}}\left[ \left\| \boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta\left(\sqrt{\bar\alpha_t} \mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\boldsymbol{\epsilon}, t\right) \right\|^2\right] \qquad (14)
\end{aligned}$$
<a id="equation_14" />


其中$t$在1和$T$之间均匀分布。$t=1$情况对应$L_0$，在离散解码器定义[公式13](#equation_13)中的积分通过高斯概率密度函数乘以区间宽度来近似，忽略了$\sigma_1^2$和边缘效应。$t\gt 1$情况对应[公式12](#equation_12)的无权重版本，类似于NCSN噪声分数匹配模型中使用的损失加权。($L_\mathcal{T}$不出现是因为正向过程方差$\beta_t$是固定的。)[算法1](#algorithm_1)展示了使用这个简化目标的完整训练过程。


由于我们的简化目标[公式14](#equation_14) 丢弃了[公式12](#equation_12)中的加权，与标准变分界相比，它是一种加权变分界，强调重建的不同方面。特别是，第 4 节中的我们的扩散过程设置导致简化目标降低对应于小$t$的损失项的权重。这些项训练网络对带有非常少量噪声的数据进行去噪，因此降低它们的权重是有益的，这样网络可以专注于在较大的$t$项上更困难的去噪任务。我们的实验将表明，这种重新加权会导致更好的样本质量。